# Public repository note

This notebook is an output-cleared code copy. It requires locally authorised data and is not runnable from the public repository alone. Green Street raw data, intermediate files, derived aggregates and outputs are not distributed.


# 06 Mapping and Spatial Analysis

This compact notebook contains the figures currently ready for the dissertation discussion with Gavin:

- MSOA coverage and Green Street/OpenLocal source validation;
- the H1 station-to-MSOA research design;
- residential-origin retail outcomes and the corresponding MSOA models.

LAD diagnostics, H2 and provisional H3 figures are deliberately omitted here. They remain available in earlier outputs and can be rebuilt in dedicated robustness or hypothesis notebooks after the methods are finalised.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import textwrap
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.patches import Patch
from scipy import stats
import statsmodels.api as sm
from shapely import wkt

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 160)

BASE = Path(os.environ.get("DISSERTATION_WORKSPACE", Path.cwd().resolve()))
HYP_DIR = BASE / "outputs" / "restricted_hypothesis_testing"
RECON_DIR = BASE / "outputs" / "restricted_source_reconciliation"
MSOA_DIR = BASE / "outputs" / "restricted_msoa_origin_exposure_analysis"
OUTPUT_DIR = BASE / "outputs" / "restricted_mapping_and_spatial_analysis"
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "font.size": 9,
    "axes.titlesize": 12,
    "axes.labelsize": 9,
})

PALETTE = {
    "workplace": "#e4b0a8",
    "workplace_dark": "#a86258",
    "origin": "#a8c2d6",
    "origin_dark": "#527d9c",
    "teal": "#7aa99a",
    "green": "#8fae82",
    "gold": "#d6b36a",
    "purple": "#a997c8",
    "neutral": "#e7e5df",
    "neutral_dark": "#666666",
    "ink": "#263238",
}

OUTPUT_DIR

## 2. Load Spatial Layers and Analysis Panels

In [ ]:
files = {
    "lad_boundaries": BASE / "Local_Authority_Districts_May_2024_Boundaries_UK_BGC_3503156029110784919.geojson",
    "office_markets": BASE / "London_Office_Markets_V1.geojson",
    "top100_stations": BASE / "Top100_Master_Spatial_OD_Sheet_Clean.csv",
    "station_shock_metrics": BASE / "outputs" / "restricted_h1_fine_grained_analysis" / "tfl_all_station_commuter_shock.csv",
    "od_workplace": BASE / "ODWP01EW_LTLA.csv",
    "change_panel": HYP_DIR / "hypothesis_change_panel_lad.csv",
    "destination_exposure": HYP_DIR / "tfl_destination_lad_exposure.csv",
    "origin_exposure": HYP_DIR / "ons_od_origin_exposure.csv",
    "validation_panel": RECON_DIR / "source_validation_lad_year_panel.csv",
    "coverage_lad": RECON_DIR / "source_reconciliation_coverage_by_lad.csv",
    "greenstreet_lad_year": RECON_DIR / "greenstreet_lad_year_indicators.csv",
    "openlocal_lad_year": RECON_DIR / "openlocal_lad_year_indicators.csv",
}

missing = [k for k, p in files.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing inputs: {missing}")

lad_all = gpd.read_file(files["lad_boundaries"]).to_crs("EPSG:27700")
london_lad = lad_all[lad_all["LAD24CD"].str.startswith("E09", na=False)].copy()

office = gpd.read_file(files["office_markets"]).to_crs("EPSG:27700")
submarket_crosswalk = {
    "West End": ["Mayfair", "Soho", "St James's", "Covent Garden", "Fitzrovia", "North of Oxford Street", "Paddington", "Knightsbridge", "Victoria"],
    "City": ["City Core"],
    "Tech Belt & Midtown": ["Midtown", "Bloomsbury", "Clerkenwell", "Euston", "Kings Cross", "Shoreditch", "Camden", "Aldgate & Whitechapel"],
    "Canary Wharf": ["Canary Wharf"],
    "Southbank": ["Southbank", "Waterloo", "Vauxhall, Nine Elms and Battersea"],
}
market_to_group = {market: group for group, markets in submarket_crosswalk.items() for market in markets}
office["study_submarket"] = office["Market"].map(market_to_group).fillna("Outside core / comparison")
office["inside_core_submarket"] = office["study_submarket"].ne("Outside core / comparison")
core_office = office[office["inside_core_submarket"]].copy()

top100 = pd.read_csv(files["top100_stations"])
station_metrics = pd.read_csv(
    files["station_shock_metrics"],
    usecols=["clean_name", "2019_Midweek", "cumulative_collapse_score"],
)
top100 = (
    top100.drop(columns=["cumulative_collapse_score"], errors="ignore")
    .merge(station_metrics, on="clean_name", how="left", validate="one_to_one")
)
if top100[["2019_Midweek", "cumulative_collapse_score"]].isna().any().any():
    raise ValueError("Top-100 stations are missing 2019 activity or commuter-shock metrics.")
top100 = top100.dropna(subset=["geometry"]).copy()
top100["geometry"] = top100["geometry"].astype(str).apply(wkt.loads)
stations_gdf = gpd.GeoDataFrame(top100, geometry="geometry", crs="EPSG:4326").to_crs("EPSG:27700")

change_panel = pd.read_csv(files["change_panel"])
destination_exposure = pd.read_csv(files["destination_exposure"])
origin_exposure = pd.read_csv(files["origin_exposure"])
od_workplace = pd.read_csv(files["od_workplace"])
validation_panel = pd.read_csv(files["validation_panel"])
coverage_lad = pd.read_csv(files["coverage_lad"])
greenstreet_lad_year = pd.read_csv(files["greenstreet_lad_year"])
openlocal_lad_year = pd.read_csv(files["openlocal_lad_year"])

print("London LADs:", len(london_lad))
print("Core office market polygons:", len(core_office))
print("Affected station records:", len(stations_gdf))
print("Change panel LADs:", len(change_panel))
display(change_panel[["LAD24CD", "LAD24NM", "affected_stations", "mean_destination_collapse"]].head())

## 3. Mapping Helper Functions

In [ ]:
def attach_to_lads(df, code_col="LAD24CD", keep_cols=None):
    if keep_cols is None:
        keep_cols = [c for c in df.columns if c != code_col]
    merged = london_lad.merge(
        df[[code_col] + keep_cols].drop_duplicates(code_col),
        left_on="LAD24CD",
        right_on=code_col,
        how="left",
    )
    if code_col != "LAD24CD" and code_col in merged.columns:
        merged = merged.drop(columns=[code_col])
    return merged


def clean_axis(ax):
    ax.set_axis_off()
    ax.set_aspect("equal")


def draw_spatial_context(ax):
    office.boundary.plot(ax=ax, color="#8f8f8f", linewidth=0.28, alpha=0.45)
    core_office.boundary.plot(ax=ax, color="#222222", linewidth=0.85, alpha=0.9)
    london_lad.boundary.plot(ax=ax, color="#b5b5b5", linewidth=0.25)


def plot_lad_map(gdf, column, title, filename, cmap="viridis", diverging=False, label=None, missing_label="No data"):
    fig, ax = plt.subplots(figsize=(8.2, 7.2))
    london_lad.plot(ax=ax, color="#f2f2f2", edgecolor="#d8d8d8", linewidth=0.35)

    plot_kwargs = {
        "column": column,
        "ax": ax,
        "cmap": cmap,
        "legend": True,
        "edgecolor": "#ffffff",
        "linewidth": 0.35,
        "missing_kwds": {"color": "#eeeeee", "edgecolor": "#d0d0d0", "hatch": "///", "label": missing_label},
    }
    if diverging:
        vals = gdf[column].replace([np.inf, -np.inf], np.nan).dropna()
        if len(vals):
            vmax = max(abs(vals.quantile(0.02)), abs(vals.quantile(0.98)))
            if vmax == 0:
                vmax = max(abs(vals.min()), abs(vals.max()), 1)
            plot_kwargs["norm"] = TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax)
    gdf.plot(**plot_kwargs)
    draw_spatial_context(ax)
    ax.set_title(title, loc="left", pad=10)
    if label:
        ax.text(
            0.01,
            0.98,
            textwrap.fill(label, width=62),
            transform=ax.transAxes,
            fontsize=8,
            color="#555555",
            va="top",
            ha="left",
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.72, "pad": 3},
        )
    clean_axis(ax)
    fig.tight_layout()
    out = FIG_DIR / filename
    fig.savefig(out, bbox_inches="tight")
    plt.show()
    return out


def plot_lad_map_on_axis(ax, gdf, column, title, cmap="viridis", diverging=False, legend_label=None, missing_label="No data"):
    london_lad.plot(ax=ax, color="#f2f2f2", edgecolor="#d8d8d8", linewidth=0.32)
    plot_kwargs = {
        "column": column,
        "ax": ax,
        "cmap": cmap,
        "legend": True,
        "legend_kwds": {"shrink": 0.72, "pad": 0.02},
        "edgecolor": "#ffffff",
        "linewidth": 0.32,
        "missing_kwds": {"color": "#eeeeee", "edgecolor": "#d0d0d0", "hatch": "///", "label": missing_label},
    }
    if legend_label:
        plot_kwargs["legend_kwds"]["label"] = legend_label
    if diverging:
        vals = gdf[column].replace([np.inf, -np.inf], np.nan).dropna()
        if len(vals):
            vmax = max(abs(vals.quantile(0.02)), abs(vals.quantile(0.98)))
            if vmax == 0:
                vmax = max(abs(vals.min()), abs(vals.max()), 1)
            plot_kwargs["norm"] = TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax)
    gdf.plot(**plot_kwargs)
    draw_spatial_context(ax)
    ax.set_title(title, loc="left", pad=8, fontsize=11)
    clean_axis(ax)


def plot_categorical_lad_map(gdf, column, style_map, title, filename, label=None, annotate_categories=None):
    annotate_categories = set(annotate_categories or [])
    fig, ax = plt.subplots(figsize=(8.2, 7.2))
    london_lad.plot(ax=ax, color="#f2f2f2", edgecolor="#d8d8d8", linewidth=0.35)
    present_categories = []
    for category, style in style_map.items():
        sub = gdf[gdf[column].eq(category)]
        if len(sub):
            present_categories.append(category)
            if isinstance(style, str):
                style = {"facecolor": style}
            sub.plot(
                ax=ax,
                color=style.get("facecolor", "#d9d9d9"),
                edgecolor=style.get("edgecolor", "white"),
                linewidth=style.get("linewidth", 0.45),
                hatch=style.get("hatch", None),
            )
            if category in annotate_categories and "LAD24NM" in sub.columns:
                for _, row in sub.iterrows():
                    pt = row.geometry.representative_point()
                    ax.text(
                        pt.x,
                        pt.y,
                        row["LAD24NM"],
                        ha="center",
                        va="center",
                        fontsize=7.5,
                        color="#2f2f2f",
                        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.72, "pad": 1.5},
                    )
    draw_spatial_context(ax)
    handles = []
    for category in present_categories:
        style = style_map[category]
        if isinstance(style, str):
            style = {"facecolor": style}
        count = int(gdf[column].eq(category).sum())
        handles.append(
            Patch(
                facecolor=style.get("facecolor", "#d9d9d9"),
                edgecolor=style.get("edgecolor", "none"),
                hatch=style.get("hatch", None),
                label=f"{category} (n={count})",
            )
        )
    ax.legend(handles=handles, frameon=False, loc="lower left", fontsize=8)
    ax.set_title(title, loc="left", pad=10)
    if label:
        ax.text(
            0.01,
            0.98,
            textwrap.fill(label, width=62),
            transform=ax.transAxes,
            fontsize=8,
            color="#555555",
            va="top",
            ha="left",
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.72, "pad": 3},
        )
    clean_axis(ax)
    fig.tight_layout()
    out = FIG_DIR / filename
    fig.savefig(out, bbox_inches="tight")
    plt.show()
    return out


def plot_station_map(stations, title, filename):
    fig, ax = plt.subplots(figsize=(8.2, 7.2))
    london_lad.plot(ax=ax, color="#f7f7f7", edgecolor="#d8d8d8", linewidth=0.35)
    core_office.plot(ax=ax, color="#edf2f7", edgecolor="#333333", linewidth=0.55, alpha=0.55)
    stations.plot(
        ax=ax,
        column="cumulative_collapse_score",
        cmap="YlGnBu",
        markersize=18 + 45 * stations["cumulative_collapse_score"].rank(pct=True),
        alpha=0.78,
        legend=True,
        edgecolor="white",
        linewidth=0.25,
    )
    draw_spatial_context(ax)
    ax.set_title(title, loc="left", pad=10)
    clean_axis(ax)
    fig.tight_layout()
    out = FIG_DIR / filename
    fig.savefig(out, bbox_inches="tight")
    plt.show()
    return out


def scatter_with_fit(df, x, y, title, filename, hue=None):
    clean = df[[x, y] + ([hue] if hue else [])].replace([np.inf, -np.inf], np.nan).dropna(subset=[x, y])
    fig, ax = plt.subplots(figsize=(6.2, 5))
    if hue and hue in clean.columns:
        categories = clean[hue].astype(str).unique()
        cmap = plt.get_cmap("tab10")
        for i, cat in enumerate(categories):
            sub = clean[clean[hue].astype(str).eq(cat)]
            ax.scatter(sub[x], sub[y], s=52, alpha=0.75, label=cat, color=cmap(i % 10))
        ax.legend(frameon=False, fontsize=8)
    else:
        ax.scatter(clean[x], clean[y], s=52, alpha=0.75, color="#3267a8")
    if len(clean) >= 3:
        coef, intercept = np.polyfit(clean[x], clean[y], 1)
        xs = np.linspace(clean[x].min(), clean[x].max(), 100)
        ax.plot(xs, intercept + coef * xs, color="black", linewidth=1)
        r = clean[[x, y]].corr(method="pearson").iloc[0, 1]
        ax.text(0.02, 0.98, f"LADs={len(clean)}, r={r:.2f}", transform=ax.transAxes, va="top", ha="left", fontsize=8)
    ax.axhline(0, color="#999999", linewidth=0.7, alpha=0.6)
    ax.axvline(0, color="#999999", linewidth=0.7, alpha=0.6)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(title, loc="left")
    ax.grid(alpha=0.22)
    fig.tight_layout()
    out = FIG_DIR / filename
    fig.savefig(out, bbox_inches="tight")
    plt.show()
    return out


def scatter_on_axis(ax, df, x, y, title, xlabel, ylabel, color="#3267a8"):
    clean = df[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    ax.scatter(clean[x], clean[y], s=48, alpha=0.78, color=color, edgecolor="white", linewidth=0.35)
    if len(clean) >= 3 and clean[x].nunique() > 1:
        coef, intercept = np.polyfit(clean[x], clean[y], 1)
        xs = np.linspace(clean[x].min(), clean[x].max(), 100)
        ax.plot(xs, intercept + coef * xs, color="black", linewidth=1.05)
        pearson_r, pearson_p = stats.pearsonr(clean[x], clean[y])
        ax.text(
            0.02,
            0.98,
            f"LADs={len(clean)}; r={pearson_r:.2f}; {fmt_p(pearson_p)}",
            transform=ax.transAxes,
            va="top",
            ha="left",
            fontsize=8,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.72, "pad": 2.5},
        )
    ax.axhline(0, color="#999999", linewidth=0.7, alpha=0.65)
    ax.axvline(0, color="#999999", linewidth=0.7, alpha=0.65)
    ax.set_title(title, loc="left", fontsize=10.5)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.22)


def pct_change(post, base):
    base = pd.Series(base).replace(0, np.nan)
    return (pd.Series(post) - base) / base


def fmt_p(p):
    if pd.isna(p):
        return ""
    return "p<0.001" if p < 0.001 else f"p={p:.3f}"


def build_post_pandemic_features(df, id_col, metrics, prefix=None):
    prefix = prefix or ""
    base = (
        df[df["year"].eq(2019)][[id_col] + metrics]
        .drop_duplicates(id_col)
        .rename(columns={metric: f"{metric}_baseline_2019" for metric in metrics})
    )
    post = (
        df[df["year"].isin([2023, 2024, 2025])]
        .groupby(id_col, as_index=False)[metrics]
        .mean()
        .rename(columns={metric: f"{metric}_post_mean_2023_2025" for metric in metrics})
    )
    out = base.merge(post, on=id_col, how="outer")
    for metric in metrics:
        base_col = f"{metric}_baseline_2019"
        post_col = f"{metric}_post_mean_2023_2025"
        out[f"{metric}_change_2019_post_mean"] = out[post_col] - out[base_col]
        out[f"{metric}_pct_change_2019_post_mean"] = pct_change(out[post_col], out[base_col])
    if id_col != "LAD24CD":
        out = out.rename(columns={id_col: "LAD24CD"})
    return out


def run_ols(df, x, y, model, sample, outcome_label, predictor_label):
    clean = df[["LAD24CD", "LAD24NM", x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(clean) < 4 or clean[x].nunique() < 2:
        return None, clean
    X = sm.add_constant(clean[x])
    fitted = sm.OLS(clean[y], X).fit(cov_type="HC3")
    pearson_r, pearson_p = stats.pearsonr(clean[x], clean[y])
    spearman_r, spearman_p = stats.spearmanr(clean[x], clean[y])
    row = {
        "model": model,
        "sample": sample,
        "predictor": predictor_label,
        "outcome": outcome_label,
        "n": int(fitted.nobs),
        "beta": fitted.params[x],
        "robust_se": fitted.bse[x],
        "t_value": fitted.tvalues[x],
        "p_value": fitted.pvalues[x],
        "r_squared": fitted.rsquared,
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_r": spearman_r,
        "spearman_p": spearman_p,
    }
    residuals = clean[["LAD24CD", "LAD24NM"]].copy()
    residuals["model"] = model
    residuals["residual"] = fitted.resid
    residuals["fitted"] = fitted.fittedvalues
    residuals["observed"] = clean[y].values
    return row, residuals


def queen_neighbors(gdf):
    gdf = gdf.reset_index(drop=True)
    geoms = list(gdf.geometry)
    neighbors = {i: [] for i in range(len(gdf))}
    for i, geom_i in enumerate(geoms):
        if geom_i is None or geom_i.is_empty:
            continue
        for j in range(i + 1, len(gdf)):
            geom_j = geoms[j]
            if geom_j is None or geom_j.is_empty:
                continue
            if geom_i.touches(geom_j):
                neighbors[i].append(j)
                neighbors[j].append(i)
    return neighbors


def morans_i(values, neighbors, permutations=999, seed=42):
    values = np.asarray(values, dtype=float)
    n = len(values)
    z = values - np.nanmean(values)
    denom = np.nansum(z ** 2)
    link_count = sum(len(v) for v in neighbors.values())
    if n < 4 or denom == 0 or link_count == 0:
        return {"moran_i": np.nan, "z_score": np.nan, "p_value": np.nan, "n": n, "links": link_count}
    cross = 0.0
    for i, js in neighbors.items():
        for j in js:
            cross += z[i] * z[j]
    observed = (n / link_count) * (cross / denom)
    rng = np.random.default_rng(seed)
    sims = []
    for _ in range(permutations):
        zp = rng.permutation(z)
        cross_p = 0.0
        for i, js in neighbors.items():
            for j in js:
                cross_p += zp[i] * zp[j]
        sims.append((n / link_count) * (cross_p / denom))
    sims = np.asarray(sims)
    p_value = (np.sum(np.abs(sims) >= abs(observed)) + 1) / (permutations + 1)
    z_score = (observed - sims.mean()) / sims.std(ddof=1) if sims.std(ddof=1) > 0 else np.nan
    return {"moran_i": observed, "z_score": z_score, "p_value": p_value, "n": n, "links": link_count}


def save_table_image(df, title, filename, max_rows=12):
    show = df.head(max_rows).copy()
    wrap_widths = {
        "Metric": 30,
        "Value": 12,
        "Interpretation": 58,
        "outcome": 34,
        "predictor": 24,
        "sample": 20,
    }
    for col in show.columns:
        width = wrap_widths.get(col, 28)
        show[col] = show[col].astype(str).apply(lambda s: "\n".join(textwrap.wrap(s, width=width, break_long_words=False)))
    fig_height = 0.55 + 0.38 * (len(show) + 1)
    fig_width = max(8.5, 1.15 * len(show.columns) + 3.2)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis("off")
    ax.set_title(title, loc="left", pad=8, fontsize=12)
    if len(show.columns) == 3:
        col_widths = [0.31, 0.14, 0.55]
    else:
        col_widths = [1 / len(show.columns)] * len(show.columns)
    table = ax.table(
        cellText=show.values,
        colLabels=show.columns,
        cellLoc="left",
        colLoc="left",
        loc="center",
        colWidths=col_widths,
    )
    table.auto_set_font_size(False)
    table.set_fontsize(7.5)
    table.scale(1, 1.28)
    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor("#d0d0d0")
        cell.set_linewidth(0.35)
        if row == 0:
            cell.set_facecolor("#e9edf2")
            cell.set_text_props(weight="bold")
        elif row % 2 == 0:
            cell.set_facecolor("#f8f9fb")
    fig.tight_layout()
    out = FIG_DIR / filename
    fig.savefig(out, bbox_inches="tight")
    plt.show()
    return out


figure_records = []

def add_record(fig_id, filename, theme, description, chapter_candidate):
    figure_records.append({
        "figure_id": fig_id,
        "filename": filename,
        "theme": theme,
        "description": description,
        "chapter_candidate": chapter_candidate,
    })

## 4. MSOA Data Coverage


In [ ]:
target_years = [2019, 2023, 2024, 2025]

# MSOA-level source coverage and validation for the refined H1 geography.
msoa_boundary_path = MSOA_DIR / "london_msoa_2021_boundaries.geojson"
msoa_openlocal_year_path = MSOA_DIR / "openlocal_msoa_year_indicators.csv"
greenstreet_property_path = RECON_DIR / "greenstreet_property_year_panel.csv"

def safe_corr(df, cols, method):
    clean = df[list(cols)].replace([np.inf, -np.inf], np.nan).dropna()
    if len(clean) < 3:
        return np.nan
    return clean.corr(method=method).iloc[0, 1]

msoa_validation = pd.DataFrame()
if msoa_boundary_path.exists() and msoa_openlocal_year_path.exists() and greenstreet_property_path.exists():
    london_msoa = gpd.read_file(msoa_boundary_path).to_crs("EPSG:27700")
    openlocal_msoa_year = pd.read_csv(msoa_openlocal_year_path)
    greenstreet_property_year = pd.read_csv(greenstreet_property_path, low_memory=False)
    for col in ["LATITUDE", "LONGITUDE", "NUMBER_UNIT", "RATE_VACANT", "RATE_VACANT_RETAIL", "SCORE_HEALTH_INDEX"]:
        if col in greenstreet_property_year.columns:
            greenstreet_property_year[col] = pd.to_numeric(greenstreet_property_year[col], errors="coerce")
    gs_points = greenstreet_property_year.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()
    gs_gdf = gpd.GeoDataFrame(
        gs_points,
        geometry=gpd.points_from_xy(gs_points["LONGITUDE"], gs_points["LATITUDE"]),
        crs="EPSG:4326",
    ).to_crs("EPSG:27700")
    gs_msoa_joined = gpd.sjoin(
        gs_gdf,
        london_msoa[["MSOA21CD", "MSOA21NM", "geometry"]],
        how="inner",
        predicate="within",
    ).drop(columns=["index_right"], errors="ignore")
    gs_msoa_joined["gs_unit_weight"] = gs_msoa_joined["NUMBER_UNIT"].fillna(0).clip(lower=0)
    gs_msoa_joined["gs_vac_num"] = gs_msoa_joined["RATE_VACANT"] * gs_msoa_joined["gs_unit_weight"]
    greenstreet_msoa_year = (
        gs_msoa_joined.groupby(["MSOA21CD", "MSOA21NM", "year"], as_index=False)
        .agg(
            gs_properties=("PROPERTY_ID", "nunique"),
            gs_units=("NUMBER_UNIT", "sum"),
            gs_vac_num=("gs_vac_num", "sum"),
            gs_weight=("gs_unit_weight", "sum"),
            gs_mean_vacancy=("RATE_VACANT", "mean"),
            gs_mean_health_index=("SCORE_HEALTH_INDEX", "mean"),
        )
    )
    greenstreet_msoa_year["gs_weighted_vacancy"] = (
        greenstreet_msoa_year["gs_vac_num"] / greenstreet_msoa_year["gs_weight"].replace(0, np.nan)
    )
    greenstreet_msoa_year["gs_weighted_vacancy"] = greenstreet_msoa_year["gs_weighted_vacancy"].fillna(greenstreet_msoa_year["gs_mean_vacancy"])
    greenstreet_msoa_year.to_csv(OUTPUT_DIR / "greenstreet_msoa_year_indicators.csv", index=False)

    msoa_validation = greenstreet_msoa_year.merge(
        openlocal_msoa_year[["MSOA21CD", "MSOA21NM", "year", "retail_units", "vacancy_proxy"]],
        on=["MSOA21CD", "MSOA21NM", "year"],
        how="inner",
    )
    msoa_validation = msoa_validation[msoa_validation["year"].isin([2019, 2023, 2024, 2025])].copy()
    msoa_validation["vacancy_gap_gs_minus_ol"] = msoa_validation["gs_weighted_vacancy"] - msoa_validation["vacancy_proxy"]
    msoa_validation.to_csv(OUTPUT_DIR / "source_validation_msoa_year_panel.csv", index=False)

    gs_msoa_coverage = greenstreet_msoa_year.groupby("MSOA21CD").agg(gs_years=("year", "nunique"), mean_gs_units=("gs_units", "mean")).reset_index()
    ol_msoa_coverage = openlocal_msoa_year.groupby("MSOA21CD").agg(ol_years=("year", "nunique"), mean_ol_retail_units=("retail_units", "mean")).reset_index()
    msoa_coverage = (
        london_msoa[["MSOA21CD", "MSOA21NM", "geometry"]]
        .merge(gs_msoa_coverage, on="MSOA21CD", how="left")
        .merge(ol_msoa_coverage, on="MSOA21CD", how="left")
    )
    for col in ["gs_years", "ol_years", "mean_gs_units", "mean_ol_retail_units"]:
        msoa_coverage[col] = msoa_coverage[col].fillna(0)
    msoa_coverage["coverage_status"] = np.select(
        [
            msoa_coverage["gs_years"].gt(0) & msoa_coverage["ol_years"].gt(0),
            msoa_coverage["gs_years"].gt(0),
            msoa_coverage["ol_years"].gt(0),
        ],
        ["Both sources", "Green Street only", "OpenLocal only"],
        default="No analysis-ready coverage",
    )
    msoa_coverage.drop(columns="geometry").to_csv(OUTPUT_DIR / "source_coverage_msoa.csv", index=False)

    msoa_colors = {
        "Both sources": "#6f9f8f",
        "Green Street only": PALETTE["workplace"],
        "OpenLocal only": PALETTE["origin"],
        "No analysis-ready coverage": "#f0eee8",
    }
    core_by_submarket = core_office.dissolve(by="study_submarket").reset_index()

    fig, ax = plt.subplots(figsize=(9.2, 7.05))
    for status, color in msoa_colors.items():
        sub = msoa_coverage[msoa_coverage["coverage_status"].eq(status)]
        if len(sub):
            sub.plot(ax=ax, color=color, edgecolor="#f8f8f8", linewidth=0.018, alpha=0.90)
    london_lad.boundary.plot(ax=ax, color="#b8b8b8", linewidth=0.22, alpha=0.55)
    core_by_submarket.boundary.plot(ax=ax, color="#202020", linewidth=1.05, alpha=0.95)
    core_office.boundary.plot(ax=ax, color="#777777", linewidth=0.16, alpha=0.22)
    ax.set_title("MSOA Source Coverage: Green Street and OpenLocal", loc="left", pad=8)
    handles = [Patch(facecolor=color, edgecolor="white", label=status) for status, color in msoa_colors.items()]
    ax.legend(handles=handles, frameon=False, fontsize=8.4, loc="lower left")
    clean_axis(ax)
    p = FIG_DIR / "fig_01c_msoa_source_coverage_classification_map.png"
    fig.savefig(p, bbox_inches="tight")
    plt.show()
    add_record("Figure 4.x", p.name, "MSOA data coverage", "MSOA-level source coverage for Green Street and OpenLocal in the refined H1 geography.", "Chapter 4 Methodology / data")


## 5. H1 Maps: Workplace Shock and Residential Origin Exposure

In [ ]:
fig, ax = plt.subplots(figsize=(12.2, 3.25))
ax.axis("off")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

flow_nodes = [
    (0.11, 0.66, "TfL station data\n2019 baseline vs\n2023-2025"),
    (0.36, 0.66, "Affected stations\nRanked commuter-\ncollapse measure"),
    (0.62, 0.66, "Office submarket filter\nSelected affected\nworkplace stations"),
    (0.88, 0.66, "Workplace MSOAs\nStation areas linked\nto Census OD"),
    (0.77, 0.22, "Residential MSOA exposure\nCommuting flows weighted by\nactivity-weighted station shock"),
    (0.43, 0.22, "Retail outcome tests\nOpenLocal indicators and\nGreen Street outcomes"),
]

box_style = dict(boxstyle="round,pad=0.48", facecolor="#edf2f7", edgecolor="#4a5568", linewidth=1.15)
highlight_style = dict(boxstyle="round,pad=0.48", facecolor="#fff0cf", edgecolor="#a34b12", linewidth=1.15)

for i, (x, y, text) in enumerate(flow_nodes):
    ax.text(
        x,
        y,
        text,
        ha="center",
        va="center",
        fontsize=10.8,
        linespacing=1.13,
        bbox=highlight_style if i in [2, 4] else box_style,
    )

arrow_pairs = [
    ((0.20, 0.66), (0.27, 0.66)),
    ((0.46, 0.66), (0.52, 0.66)),
    ((0.72, 0.66), (0.79, 0.66)),
    ((0.88, 0.53), (0.80, 0.34)),
    ((0.68, 0.22), (0.55, 0.22)),
]
for start, end in arrow_pairs:
    ax.annotate(
        "",
        xy=end,
        xytext=start,
        arrowprops=dict(arrowstyle="->", color="#4a5568", linewidth=1.55, shrinkA=5, shrinkB=5),
    )

ax.set_title("H1 Spatial Mechanism: Station Shock to Residential MSOA Exposure", loc="left", pad=4, fontsize=15)
fig.subplots_adjust(left=0.015, right=0.985, top=0.86, bottom=0.02)
p = FIG_DIR / "fig_03a_h1_spatial_mechanism_flow.png"
fig.savefig(p, bbox_inches="tight", pad_inches=0.03, dpi=300)
plt.show()
add_record("Figure 4.x / 5.x", p.name, "H1", "Mechanism linking affected workplace stations to residential-origin MSOA exposure.", "Chapter 4 Methodology or Chapter 5 Results introduction")

MAIN_SUBMARKET_BUFFER_M = 500
BUFFER_SENSITIVITY_M = [300, 500, 800]
ORIGIN_TOP_N_SENSITIVITY = [15, 20]

core_by_submarket = core_office.dissolve(by="study_submarket").reset_index()[["study_submarket", "geometry"]]
distance_matrix = pd.DataFrame({
    row["study_submarket"]: stations_gdf.geometry.distance(row.geometry)
    for _, row in core_by_submarket.iterrows()
})
stations_enriched = stations_gdf.copy()
stations_enriched["nearest_core_submarket"] = distance_matrix.idxmin(axis=1)
stations_enriched["distance_to_core_submarket_m"] = distance_matrix.min(axis=1)
stations_enriched["inside_core_submarket"] = stations_enriched["distance_to_core_submarket_m"].le(1e-6)

def station_subset_for_buffer(buffer_m):
    return stations_enriched[
        stations_enriched["distance_to_core_submarket_m"].le(buffer_m)
        & stations_enriched["LAD24CD"].astype(str).str.startswith("E09")
    ].copy()

selected_stations = station_subset_for_buffer(MAIN_SUBMARKET_BUFFER_M)
selected_station_names = set(selected_stations["clean_name"].dropna())

fig, ax = plt.subplots(figsize=(8.2, 7.0))
london_lad.plot(ax=ax, color="#f6f6f4", edgecolor="#d0d0d0", linewidth=0.28)
core_office.plot(ax=ax, color="none", edgecolor=PALETTE["ink"], linewidth=0.85, alpha=0.9)
stations_enriched.plot(ax=ax, color="#cfcfcf", markersize=20, alpha=0.55, edgecolor="white", linewidth=0.2)
selected_stations.plot(
    ax=ax,
    column="cumulative_collapse_score",
    cmap="YlGnBu",
    markersize=34 + 45 * selected_stations["cumulative_collapse_score"].rank(pct=True),
    alpha=0.88,
    legend=True,
    edgecolor="white",
    linewidth=0.25,
)
draw_spatial_context(ax)
ax.legend(
    handles=[
        Patch(facecolor="#cfcfcf", edgecolor="white", label="Top-100 affected station"),
        Patch(facecolor=PALETTE["origin"], edgecolor="white", label=f"Selected station within {MAIN_SUBMARKET_BUFFER_M}m buffer"),
    ],
    frameon=False,
    loc="lower left",
    fontsize=8,
)
ax.set_title("H1 Workplace Station Selection within Core Office Submarkets", loc="left", pad=10)
ax.text(
    0.01,
    0.98,
    textwrap.fill("Selected workplace stations are the overlap between TfL affected-station evidence and the pre-defined office-submarket spatial frame.", width=70),
    transform=ax.transAxes,
    fontsize=8,
    color="#555555",
    va="top",
    ha="left",
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.72, "pad": 3},
)
clean_axis(ax)
fig.tight_layout()
p = FIG_DIR / "fig_03b_h1_selected_workplace_station_buffer_map.png"
fig.savefig(p, bbox_inches="tight")
plt.show()
add_record("Figure 4.x / 5.x", p.name, "H1", "Selected workplace stations formed by intersecting TfL affected stations with the core office-submarket buffer.", "Chapter 4 Methodology or Chapter 5 Results")

# Convert the two pre-filter screening maps from LADs to MSOAs.
london_msoa_screen = gpd.read_file(MSOA_DIR / "london_msoa_2021_boundaries.geojson").to_crs("EPSG:27700")
london_top100_stations = stations_gdf[
    stations_gdf["LAD24CD"].astype(str).str.startswith("E09", na=False)
].copy()
all_station_msoa = gpd.sjoin(
    london_top100_stations[["clean_name", "2019_Midweek", "cumulative_collapse_score", "geometry"]],
    london_msoa_screen[["MSOA21CD", "MSOA21NM", "geometry"]],
    how="left",
    predicate="within",
).drop(columns=["index_right"], errors="ignore")
if all_station_msoa["MSOA21CD"].isna().any():
    matched = all_station_msoa[all_station_msoa["MSOA21CD"].notna()].copy()
    missing = all_station_msoa[all_station_msoa["MSOA21CD"].isna()].drop(
        columns=["MSOA21CD", "MSOA21NM"],
        errors="ignore",
    )
    nearest = gpd.sjoin_nearest(
        missing,
        london_msoa_screen[["MSOA21CD", "MSOA21NM", "geometry"]],
        how="left",
        distance_col="nearest_msoa_distance_m",
    ).drop(columns=["index_right"], errors="ignore")
    all_station_msoa = pd.concat([matched, nearest], ignore_index=True)

all_station_msoa["weighted_shock_numerator"] = (
    all_station_msoa["cumulative_collapse_score"]
    * all_station_msoa["2019_Midweek"].clip(lower=0)
)
all_workplace_msoa_targets = (
    pd.DataFrame(all_station_msoa.drop(columns="geometry"))
    .groupby(["MSOA21CD", "MSOA21NM"], as_index=False)
    .agg(
        mean_station_shock=("cumulative_collapse_score", "mean"),
        weighted_shock_numerator=("weighted_shock_numerator", "sum"),
        baseline_station_volume=("2019_Midweek", "sum"),
        affected_station_count=("clean_name", "nunique"),
    )
    .rename(columns={"MSOA21CD": "workplace_msoa", "MSOA21NM": "workplace_msoa_name"})
)
all_workplace_msoa_targets["workplace_msoa_station_shock"] = (
    all_workplace_msoa_targets["weighted_shock_numerator"]
    / all_workplace_msoa_targets["baseline_station_volume"].replace(0, np.nan)
)
all_workplace_msoa_targets.to_csv(
    OUTPUT_DIR / "h1_all_top100_workplace_msoa_targets.csv",
    index=False,
)
all_workplace_codes = set(all_workplace_msoa_targets["workplace_msoa"])
all_workplace_map = london_msoa_screen.merge(
    all_workplace_msoa_targets,
    left_on="MSOA21CD",
    right_on="workplace_msoa",
    how="left",
)

workplace_vmin = all_workplace_map["workplace_msoa_station_shock"].min()
workplace_vmax = all_workplace_map["workplace_msoa_station_shock"].max()
fig, ax = plt.subplots(figsize=(11.2, 6.2))
london_msoa_screen.plot(ax=ax, color="#f1f0ed", edgecolor="white", linewidth=0.04)
all_workplace_map[all_workplace_map["workplace_msoa_station_shock"].notna()].plot(
    ax=ax,
    column="workplace_msoa_station_shock",
    cmap="Oranges",
    vmin=workplace_vmin,
    vmax=workplace_vmax,
    edgecolor="white",
    linewidth=0.12,
)
london_top100_stations.plot(ax=ax, color="#30363a", markersize=6, alpha=0.42)
core_office.boundary.plot(ax=ax, color="#50565a", linewidth=0.50, alpha=0.72)
london_lad.boundary.plot(ax=ax, color="#9ca3a6", linewidth=0.28, alpha=0.72)
ax.set_title("Top-100 Affected Stations and Their Corresponding Workplace MSOAs", loc="left", pad=5, fontsize=14)
clean_axis(ax)
workplace_sm = plt.cm.ScalarMappable(cmap="Oranges", norm=plt.Normalize(workplace_vmin, workplace_vmax))
workplace_sm.set_array([])
cax = fig.add_axes([0.31, 0.075, 0.38, 0.020])
cb = fig.colorbar(workplace_sm, cax=cax, orientation="horizontal")
cb.set_label("2019 activity-weighted commuter-shock score", fontsize=8.5, labelpad=2)
cb.ax.tick_params(labelsize=7, length=2)
fig.subplots_adjust(left=0.015, right=0.985, top=0.89, bottom=0.12)
p = FIG_DIR / "fig_04_h1_workplace_msoa_commuter_shock_map.png"
fig.savefig(p, bbox_inches="tight", pad_inches=0.035, dpi=300)
plt.show()
add_record(
    "Figure 4.x",
    p.name,
    "H1 MSOA screening",
    "All workplace MSOAs represented by the Top-100 affected stations before submarket filtering.",
    "Chapter 4 Methodology",
)

od_cols = [
    "Middle layer Super Output Areas code",
    "Middle layer Super Output Areas label",
    "MSOA of workplace code",
    "MSOA of workplace label",
    "Place of work indicator (4 categories) code",
    "Count",
]
od_msoa_all = pd.read_csv(BASE / "ODWP01EW_MSOA.csv", usecols=od_cols).rename(columns={
    "Middle layer Super Output Areas code": "origin_msoa",
    "Middle layer Super Output Areas label": "origin_msoa_name",
    "MSOA of workplace code": "workplace_msoa",
    "MSOA of workplace label": "workplace_msoa_name",
    "Place of work indicator (4 categories) code": "workplace_indicator_code",
    "Count": "commuters",
})
od_msoa_all["commuters"] = pd.to_numeric(od_msoa_all["commuters"], errors="coerce").fillna(0)
london_msoa_codes = set(london_msoa_screen["MSOA21CD"])
od_msoa_all = od_msoa_all[
    od_msoa_all["workplace_indicator_code"].eq(3)
    & od_msoa_all["origin_msoa"].isin(london_msoa_codes)
    & od_msoa_all["workplace_msoa"].isin(all_workplace_codes)
].merge(
    all_workplace_msoa_targets[["workplace_msoa", "workplace_msoa_station_shock"]],
    on="workplace_msoa",
    how="inner",
)
od_msoa_all["weighted_flow"] = od_msoa_all["commuters"] * od_msoa_all["workplace_msoa_station_shock"]
all_origin_msoa_exposure = (
    od_msoa_all.groupby(["origin_msoa", "origin_msoa_name"], as_index=False)
    .agg(
        commuters_to_top100_workplaces=("commuters", "sum"),
        od_weighted_msoa_exposure_sum=("weighted_flow", "sum"),
        linked_workplace_msoas=("workplace_msoa", "nunique"),
    )
)
all_origin_msoa_exposure["msoa_exposure_per_1000"] = (
    all_origin_msoa_exposure["od_weighted_msoa_exposure_sum"] / 1000
)
all_origin_msoa_exposure.to_csv(
    OUTPUT_DIR / "h1_all_top100_residential_origin_msoa_exposure.csv",
    index=False,
)
all_origin_map = london_msoa_screen.merge(
    all_origin_msoa_exposure,
    left_on="MSOA21CD",
    right_on="origin_msoa",
    how="left",
)
openlocal_msoa_codes = set(
    pd.read_csv(MSOA_DIR / "openlocal_msoa_year_indicators.csv", usecols=["MSOA21CD"])["MSOA21CD"].dropna()
)
all_origin_map["has_supplied_retail_outcomes"] = all_origin_map["MSOA21CD"].isin(openlocal_msoa_codes)

fig, ax = plt.subplots(figsize=(9.4, 7.6))
london_msoa_screen.plot(ax=ax, color="#f1f0ed", edgecolor="white", linewidth=0.04)
all_origin_map[
    all_origin_map["msoa_exposure_per_1000"].gt(0)
    & all_origin_map["has_supplied_retail_outcomes"]
].plot(
    ax=ax,
    column="msoa_exposure_per_1000",
    cmap="Blues",
    legend=True,
    legend_kwds={"shrink": 0.62, "label": "Residential exposure (weighted flow / 1,000)"},
    edgecolor="white",
    linewidth=0.08,
)
unavailable_outcomes = all_origin_map[
    all_origin_map["msoa_exposure_per_1000"].gt(0)
    & ~all_origin_map["has_supplied_retail_outcomes"]
]
if len(unavailable_outcomes):
    unavailable_outcomes.plot(
        ax=ax,
        color="#f5f4f1",
        edgecolor="#9d9d9d",
        linewidth=0.16,
        hatch="///",
    )
core_office.boundary.plot(ax=ax, color=PALETTE["ink"], linewidth=0.45, alpha=0.8)
london_lad.boundary.plot(ax=ax, color="#b8b8b8", linewidth=0.25)
ax.set_title("Analysis-Eligible Residential-Origin MSOA Exposure", loc="left", pad=10)
ax.text(
    0.01,
    0.98,
    "Blue MSOAs combine ONS exposure with supplied retail outcomes. Hatching marks exposure that cannot enter the retail analysis.",
    transform=ax.transAxes,
    fontsize=8.5,
    color="#555555",
    va="top",
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.78, "pad": 3},
)
ax.legend(
    handles=[Patch(facecolor="white", edgecolor="#9d9d9d", hatch="///", label="No supplied retail outcome data")],
    frameon=False,
    fontsize=8,
    loc="lower left",
)
ax.axis("off")
p = FIG_DIR / "fig_05_h1_residential_origin_msoa_exposure_map.png"
fig.savefig(p, bbox_inches="tight")
plt.show()
add_record(
    "Figure 4.x",
    p.name,
    "H1 MSOA screening",
    "ONS-derived residential-origin exposure with supplied retail-outcome availability shown separately.",
    "Chapter 4 Methodology",
)

# Preferred H1 origin geography: selected workplace stations -> workplace MSOAs -> residential MSOAs.
msoa_boundary_path = MSOA_DIR / "london_msoa_2021_boundaries.geojson"
msoa_exposure_path = MSOA_DIR / "h1_origin_msoa_exposure.csv"
workplace_msoa_path = MSOA_DIR / "h1_selected_workplace_msoa_targets.csv"
if msoa_boundary_path.exists() and msoa_exposure_path.exists() and workplace_msoa_path.exists():
    london_msoa_h1 = gpd.read_file(msoa_boundary_path).to_crs("EPSG:27700")
    msoa_exposure_h1 = pd.read_csv(msoa_exposure_path)
    workplace_msoa_targets = pd.read_csv(workplace_msoa_path)
    msoa_h1_map = london_msoa_h1.merge(
        msoa_exposure_h1[["origin_msoa", "msoa_exposure_per_1000", "od_weighted_msoa_exposure_sum"]],
        left_on="MSOA21CD",
        right_on="origin_msoa",
        how="left",
    )
    msoa_h1_map = msoa_h1_map.merge(
        workplace_msoa_targets[["workplace_msoa", "workplace_msoa_station_shock"]],
        left_on="MSOA21CD",
        right_on="workplace_msoa",
        how="left",
    )
    msoa_h1_map["msoa_exposure_per_1000"] = msoa_h1_map["msoa_exposure_per_1000"].fillna(0)
    msoa_h1_map["is_workplace_station_msoa"] = msoa_h1_map["MSOA21CD"].isin(set(workplace_msoa_targets["workplace_msoa"]))
    openlocal_msoa_codes = set(
        pd.read_csv(MSOA_DIR / "openlocal_msoa_year_indicators.csv", usecols=["MSOA21CD"])["MSOA21CD"].dropna()
    )
    msoa_h1_map["has_supplied_retail_outcomes"] = msoa_h1_map["MSOA21CD"].isin(openlocal_msoa_codes)

    eligible_origins = msoa_h1_map[
        msoa_h1_map["msoa_exposure_per_1000"].gt(0)
        & msoa_h1_map["has_supplied_retail_outcomes"]
        & ~msoa_h1_map["is_workplace_station_msoa"]
    ].copy()
    workplace_areas = msoa_h1_map[msoa_h1_map["is_workplace_station_msoa"]].copy()
    exposure_vmax = eligible_origins["msoa_exposure_per_1000"].quantile(0.98)
    shock_vmin = workplace_areas["workplace_msoa_station_shock"].min()
    shock_vmax = workplace_areas["workplace_msoa_station_shock"].max()

    fig = plt.figure(figsize=(12.2, 6.55))
    grid = fig.add_gridspec(
        1, 2, width_ratios=[3.25, 1.30],
        left=0.015, right=0.985, top=0.82, bottom=0.13, wspace=0.025,
    )
    ax = fig.add_subplot(grid[0, 0])
    ax_zoom = fig.add_subplot(grid[0, 1])

    for map_ax in (ax, ax_zoom):
        msoa_h1_map.plot(ax=map_ax, color="#f1f3f3", edgecolor="white", linewidth=0.025)
        eligible_origins.plot(
            ax=map_ax, column="msoa_exposure_per_1000", cmap="Blues",
            vmin=0, vmax=exposure_vmax, edgecolor="white", linewidth=0.025,
        )
        workplace_areas.plot(
            ax=map_ax, column="workplace_msoa_station_shock", cmap="Oranges",
            vmin=shock_vmin, vmax=shock_vmax,
            edgecolor=PALETTE["workplace_dark"], linewidth=0.50, alpha=0.94,
        )
        london_lad.boundary.plot(ax=map_ax, color="#9ca3a6", linewidth=0.28, alpha=0.72)
        clean_axis(map_ax)

    # The main panel shows the London-wide origin pattern. The detail panel
    # carries the station and office-boundary context so the main map stays legible.
    workplace_bounds = workplace_areas.total_bounds
    zoom_margin = 4500
    ax_zoom.set_xlim(workplace_bounds[0] - zoom_margin, workplace_bounds[2] + zoom_margin)
    ax_zoom.set_ylim(workplace_bounds[1] - zoom_margin, workplace_bounds[3] + zoom_margin)
    core_office.boundary.plot(ax=ax_zoom, color="#42484c", linewidth=0.75, alpha=0.80)
    selected_stations.plot(ax=ax_zoom, color="#263238", markersize=9, alpha=0.70, edgecolor="white", linewidth=0.20)
    ax_zoom.set_title("Central London workplace detail", fontsize=10.5, pad=5)

    fig.suptitle(
        "H1 Main Spatial Framework: Workplace Shock and Residential Exposure",
        x=0.02, y=0.965, ha="left", fontsize=16,
    )
    fig.text(
        0.02, 0.895,
        "Blue shows ONS-derived residential exposure with available retail outcomes; orange shows commuter shock in selected workplace MSOAs.",
        ha="left", va="center", fontsize=9.7, color="#555555",
    )

    exposure_sm = plt.cm.ScalarMappable(cmap="Blues", norm=plt.Normalize(0, exposure_vmax))
    shock_sm = plt.cm.ScalarMappable(cmap="Oranges", norm=plt.Normalize(shock_vmin, shock_vmax))
    exposure_sm.set_array([])
    shock_sm.set_array([])
    cax_exposure = fig.add_axes([0.16, 0.060, 0.34, 0.018])
    cax_shock = fig.add_axes([0.705, 0.060, 0.20, 0.018])
    cb_exposure = fig.colorbar(exposure_sm, cax=cax_exposure, orientation="horizontal")
    cb_shock = fig.colorbar(shock_sm, cax=cax_shock, orientation="horizontal")
    cb_exposure.set_label("Residential exposure (weighted flow / 1,000)", fontsize=8.5, labelpad=2)
    cb_shock.set_label("2019 activity-weighted workplace shock", fontsize=8.5, labelpad=2)
    cb_exposure.ax.tick_params(labelsize=7, length=2)
    cb_shock.ax.tick_params(labelsize=7, length=2)

    p = FIG_DIR / "fig_04_h1_main_msoa_workplace_origin_exposure_map.png"
    fig.savefig(p, bbox_inches="tight", pad_inches=0.04, dpi=300)
    plt.show()
    add_record("Figure 5.x", p.name, "H1 MSOA", "Main H1 spatial framework showing selected workplace MSOAs and residential MSOA exposure.", "Chapter 5 Results")

## 6. H1 MSOA Retail Outcome Maps, Main Tests and Spatial Diagnostics

In [ ]:
msoa_boundary_path = MSOA_DIR / "london_msoa_2021_boundaries.geojson"
msoa_change_path = MSOA_DIR / "openlocal_msoa_2019_to_post_mean_change_indicators.csv"
workplace_msoa_path = MSOA_DIR / "h1_selected_workplace_msoa_targets.csv"

if msoa_boundary_path.exists() and msoa_change_path.exists():
    london_msoa_outcomes = gpd.read_file(msoa_boundary_path).to_crs("EPSG:27700")
    msoa_changes = pd.read_csv(msoa_change_path)
    msoa_outcome_map = london_msoa_outcomes.merge(
        msoa_changes,
        on=["MSOA21CD", "MSOA21NM"],
        how="left",
    )
    workplace_msoa_codes = (
        set(pd.read_csv(workplace_msoa_path)["workplace_msoa"].dropna())
        if workplace_msoa_path.exists()
        else set()
    )
    msoa_outcome_map["is_h1_origin_msoa"] = (
        pd.to_numeric(msoa_outcome_map["msoa_exposure_per_1000"], errors="coerce")
        .fillna(0)
        .gt(0)
    )
    msoa_outcome_map["is_h1_workplace_msoa"] = msoa_outcome_map["MSOA21CD"].isin(workplace_msoa_codes)

    outcome_specs = [
        ("retail_units_pct_change", "A. Retail unit count", "RdBu_r", "Percentage change"),
        ("total_rateable_value_pct_change", "B. Total rateable value", "RdBu_r", "Percentage change"),
        ("total_floor_area_pct_change", "C. Retail floor area", "BrBG", "Percentage change"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(12.2, 4.8))
    target = msoa_outcome_map[msoa_outcome_map["is_h1_origin_msoa"]].copy()
    workplace_outline = msoa_outcome_map[msoa_outcome_map["is_h1_workplace_msoa"]]
    for ax, (column, title, cmap, cbar_label) in zip(axes, outcome_specs):
        london_msoa_outcomes.plot(ax=ax, color="#f4f2ed", edgecolor="#d7d7d7", linewidth=0.08)
        clean_values = target[column].replace([np.inf, -np.inf], np.nan).dropna()
        vmax = float(np.nanpercentile(np.abs(clean_values), 95)) if len(clean_values) else 1.0
        vmax = vmax if vmax > 0 else 1.0
        target.plot(
            ax=ax,
            column=column,
            cmap=cmap,
            norm=TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax),
            legend=True,
            legend_kwds={"shrink": 0.58, "label": cbar_label},
            edgecolor="white",
            linewidth=0.08,
            missing_kwds={"color": "#efefef"},
        )
        if len(workplace_outline):
            workplace_outline.boundary.plot(ax=ax, color=PALETTE["workplace_dark"], linewidth=0.65)
        core_office.boundary.plot(ax=ax, color=PALETTE["ink"], linewidth=0.35, alpha=0.7)
        ax.set_title(title, loc="left", fontsize=10.5)
        ax.axis("off")
    fig.suptitle(
        "H1 Residential-Origin Retail Outcomes: 2023-2025 Mean Relative to 2019",
        x=0.02,
        ha="left",
        fontsize=13,
    )
    fig.text(
        0.02,
        0.015,
        "Mapped for residential MSOAs with positive exposure. Orange outlines identify workplace-station MSOAs.",
        fontsize=8.5,
        color="#555555",
    )
    fig.tight_layout(rect=[0, 0.035, 1, 0.96])
    p = FIG_DIR / "fig_06_h1_msoa_origin_retail_outcomes_panel.png"
    fig.savefig(p, bbox_inches="tight")
    plt.show()
    add_record(
        "Figure 5.x",
        p.name,
        "H1 MSOA",
        "Three analysis-ready MSOA retail outcomes for exposed residential origins; vacancy is audited separately in Notebook 09.",
        "Chapter 5 Results",
    )


else:
    print("MSOA Objective 1 outcome inputs not found. Run 05_MSOA_Origin_Exposure_Analysis.ipynb first.")

## 7. Export Compact Figure Manifest

In [ ]:
figure_manifest = pd.DataFrame(figure_records)
figure_manifest.to_csv(OUTPUT_DIR / "figure_manifest_compact.csv", index=False)
if "stations_enriched" in globals():
    stations_enriched.drop(columns="geometry", errors="ignore").to_csv(
        OUTPUT_DIR / "h1_top100_station_submarket_distances.csv",
        index=False,
    )
if "selected_stations" in globals():
    selected_stations.drop(columns="geometry", errors="ignore").to_csv(
        OUTPUT_DIR / "h1_selected_submarket_buffer_stations.csv",
        index=False,
    )
display(figure_manifest)
print("Saved compact figures to:", FIG_DIR)

## 8. Reading Guide

Read the notebook in three steps: first check whether Green Street and OpenLocal overlap at MSOA scale; then follow the station-to-workplace-MSOA-to-residential-MSOA mechanism; finally assess whether exposure is associated with several dimensions of retail adaptation rather than vacancy alone.